In [1]:
import pandas as pd 
import tqdm

In [2]:
df = pd.read_csv("/Users/xiang/Desktop/Advising_Bot/data/Pinecone - Sheet1.csv")

In [3]:
df['_id'] = df['_id'].astype(str) 

In [4]:
df.head()

,_id,chunk_text
0,vec_1,Maintenance of Public Order: The University wi...
1,vec_2,Campus Safety: The Advisory Committee on Campu...
2,vec_3,Student Responsibility: Students are responsib...
3,vec_4,Bias-Related Crime Prevention: The Stony Brook...
4,vec_5,Student Educational Records: The Federal Famil...


In [5]:
records = df.to_dict("records")

In [6]:
from dotenv import load_dotenv 
from pinecone import Pinecone 
import os 

load_dotenv() 
api_key = os.getenv("Pinecone_key") 
pc = Pinecone(api_key=api_key) 

In [7]:
index_name = "advising-bot"
if not pc.has_index(index_name): 
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [8]:
dense_index = pc.Index(index_name)

/opt/anaconda3/envs/newEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
namespace = "bulletin"
dense_index.upsert_records(namespace, records)

UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Fri, 13 Mar 2026 21:45:14 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '670', 'x-pinecone-response-duration-ms': '671', 'server': 'envoy'}})

In [ ]:
#query= "What is the BSMS Computer Science program"
query = "What program the is Science BSMS Computer"

results = dense_index.search(
    namespace=namespace,
    query={
        "top_k": 5,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']:
        print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")

id: vec_10 | score: 0.47  | text: Joint BS/MS Program: Computer Science majors may apply for admission to a special program that leads to a Bachelor of Science degree at the end of the fourth year and a Master of Science degree at the end of the fifth year. Students usually apply to the program in their junior year. Students must satisfy the respective requirements of both the B.S. degree and the M.S. degree, but the main advantage of the program is that nine credits may be simultaneously applied to both the under­graduate and graduate requirements. The M.S. degree can therefore be earned in less time than that re­quired by the traditional course of study. For more details about the B.S./M.S. program, see the undergraduate or graduate program director in the Department of Computer Science.
id: vec_9 | score: 0.26  | text: Specialization in Systems Software Development: The specialization in systems software development prepares students for a career in software applications development

In [11]:
from google import genai

ImportError: cannot import name 'genai' from 'google' (unknown location)